# RespiScan Pipeline: Feature Cleaning & Focal Loss Experimentation
Notebook utilitas ini digunakan untuk memisahkan file visual MFCC (file `.pt` atau gambar `.png` bertanda MFCC) agar dataset latih murni berisi gambar Scalogram. Selanjutnya, dilakukan uji coba pelatihan MobileNetV2 menggunakan **Focal Loss** sebagai loss function alternatif untuk menangani sampel batuk TBC yang sulit diklasifikasikan.

### 1. Pemisahan Otomatis File MFCC & Scalogram
Script pembersihan folder otomatis untuk memindahkan file MFCC dari direktori latih utama ke direktori backup terpisah.

In [ ]:
import os
import shutil

source_dir = 'RespiScan_Train_Ready' 
mfcc_backup_dir = 'RespiScan_MFCC_Only'

phases = ['TRAIN', 'VAL', 'TEST']
classes = ['TB', 'Healthy']

pindah_count = 0
sisa_count = 0

print("🚀 Memulai proses pembersihan & pemisahan folder dataset...")

for phase in phases:
    for cls in classes:
        current_dir = os.path.join(source_dir, phase, cls)
        backup_dir = os.path.join(mfcc_backup_dir, phase, cls)
        
        if not os.path.exists(backup_dir):
            os.makedirs(backup_dir, exist_ok=True)
            
        if os.path.exists(current_dir):
            files = os.listdir(current_dir)
            for file in files:
                if 'mfcc' in file.lower() or file.endswith('.pt'):
                    src_file = os.path.join(current_dir, file)
                    dst_file = os.path.join(backup_dir, file)
                    shutil.move(src_file, dst_file)
                    pindah_count += 1
                else:
                    sisa_count += 1

print("-" * 50)
print(f"✅ Pembersihan folder Selesai!")
print(f"📁 Total file MFCC berhasil dipisahkan : {pindah_count} file.")
print(f"🖼️ Total gambar Scalogram di folder utama: {sisa_count} file.")
print(f"Folder '{source_dir}' sekarang sudah murni dan siap dilatih.")

### 2. Implementasi Algoritma Focal Loss
Mendefinisikan kelas Focal Loss kustom secara elegan dengan penanganan bobot kelas dan parameter pemfokusan gamma.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """Algoritma Focal Loss untuk melatih model pada contoh-contoh sulit."""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha  # Bobot kelas penyeimbang
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean': return focal_loss.mean()
        elif self.reduction == 'sum': return focal_loss.sum()
        return focal_loss

print("✅ Algoritma Focal Loss berhasil diimplementasikan.")

### 3. Eksperimen Pelatihan dengan Focal Loss
Pelatihan MobileNetV2 menggunakan Focal Loss dengan scheduler ReduceLROnPlateau.

In [ ]:
# Inisialisasi Device & Dataset Dataloader
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DATA_DIR = 'RespiScan_Train_Ready'

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5), 
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.05)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# (Sisa modul latihan serupa dengan trainscal.ipynb namun menggunakan FocalLoss kustom)
print("💡 Modul eksperimen Focal Loss siap diuji untuk menaklukkan sampel batuk TBC yang sulit.")